# Phase 1 versus Phase 2 — temporal-coherence diagnostics

This notebook evaluates the scientific purpose of Phase 2 on the **validation partition only**. It does not select a model from the held-out TEST results.

For the same deterministic four-year crops, it compares the frozen Phase 1 sequence with the Phase 2 residual output using:

- MSE to the exact model-derived temporal target used during training;
- mean absolute first annual difference;
- mean absolute second annual difference;
- proportions of annual changes exceeding 2 m and 5 m;
- GEDI-supported slope, spread ratio, bias, MAE, RMSE, and R²;
- temporal-target MSE for pixels with and without an algorithmic breakpoint.

Lower temporal-change diagnostics indicate smoother annual trajectories, but they are interpreted jointly with GEDI accuracy to avoid rewarding an over-smoothed or temporally constant model.

In [ ]:
from pathlib import Path
import gc, importlib.util, json, sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
WORKFLOW_PATH = ROOT / "Source" / "Project" / "final_phase2_harmonized_workflow.py"
REGISTRY_PATH = ROOT / "Source" / "Project" / "final_selected_phase2_models.json"
FINAL_SELECTION = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))["models"]
OUT = ROOT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Temporal_Diagnostics_VAL"
OUT.mkdir(parents=True, exist_ok=True)

FORESTS = ("ifran", "maamoura", "agadir")
CROP_SIZE = 96
N_CROPS_PER_FOREST = 132
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

workflow = load_module("clarck_temporal_diagnostic_workflow", WORKFLOW_PATH)
engine = workflow.base.load_engine()
workflow.install_harmonized_build_data(engine)
print("DEVICE:", DEVICE)
print("OUTPUT:", OUT)

## Exact model and pseudo-target loading

The checkpoint is taken from the retained harmonised Phase 2 run for each forest. The frozen Phase 1 output and Phase 2 output are returned by the same two-head model. The temporal target is reconstructed by the same loss builder and site-specific (D) and (K) values used during training. The AOI support is read from channel 12 and applied to every dense diagnostic.

In [ ]:
def load_site(forest):
    candidate_id, run_dir = workflow.configure(engine, forest)
    modules = workflow.base.prepare_modules(engine)
    _, shots, records = engine.build_data(forest, modules, include_test=False)
    cfg = engine.FORESTS[forest]
    final = workflow.base.FINAL_MODELS[forest]
    selected = FINAL_SELECTION[forest]
    checkpoint = Path(selected["checkpoint"])
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    checkpoint_sha256 = workflow.base.sha256_file(checkpoint)
    if checkpoint_sha256 != selected["checkpoint_sha256"]:
        raise RuntimeError(f"{forest}: selected Phase-2 checkpoint hash mismatch")
    if final["selected_checkpoint_sha256"] != selected["checkpoint_sha256"]:
        raise RuntimeError(f"{forest}: workflow and final registry disagree")

    model = engine.fresh_model(cfg, modules)
    try:
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    except TypeError:
        state = torch.load(checkpoint, map_location="cpu")
    model.prediction_head.load_state_dict(state["prediction_head"], strict=True)
    model.eval()

    dataset = modules["B4SequenceCropDataset"](
        records["val"], shots,
        crop_size=CROP_SIZE,
        samples_per_epoch=N_CROPS_PER_FOREST,
        drop_channels=(), seed=SEED + 10_000,
        center_on_gedi=True,
        balanced_height_anchors=False,
        height_bins=cfg["height_bins"],
    )

    # prepare_modules patches this builder with the retained AOI-masked wrapper.
    from src import persistent_training_v6 as training_module
    loss, provenance = training_module.build_growth_loss(
        official_repo=engine.OFFICIAL_REPO,
        disturbance_rule="persistent_running_max",
        persistent_drop_m=final["spec"]["drop_m"],
        persistent_required_consecutive_flags=final["spec"]["K"],
        disturbance_indicator=-1.0,
        slope_min=0.0, slope_max=2.0,
        full_disturbance_window=True, use_l2=True,
        max_intercept_after_disturbance=100.0,
        disturbance_factor=1.0, no_disturbance_factor=1.0,
        slope_no_disturbance=-0.0,
    )
    return model.to(DEVICE), dataset, loss.to(DEVICE), checkpoint, provenance, selected

def regression_metrics(y, pred):
    y = np.asarray(y, float); pred = np.asarray(pred, float)
    ok = np.isfinite(y) & np.isfinite(pred)
    y, pred = y[ok], pred[ok]
    residual = pred - y
    corr = np.corrcoef(y, pred)[0, 1] if len(y) > 1 else np.nan
    slope = np.cov(y, pred, ddof=0)[0, 1] / np.var(y) if np.var(y) > 0 else np.nan
    return {
        "n_gedi": len(y), "mae": np.mean(np.abs(residual)),
        "rmse": np.sqrt(np.mean(residual**2)),
        "r2": 1 - np.sum(residual**2) / np.sum((y-y.mean())**2),
        "bias": residual.mean(), "correlation": corr, "slope": slope,
        "std_ratio": np.std(pred) / np.std(y) if np.std(y) > 0 else np.nan,
    }

## Run the fixed validation audit

Each query returns one (T=4), (96\times96) crop. Phase 1, Phase 2, the pseudo-target, the AOI mask, and sparse GEDI targets are therefore spatially and temporally aligned. Progress is printed every 20 crops.

In [ ]:
summary_rows, crop_rows, change_rows, gedi_rows = [], [], [], []

for forest in FORESTS:
    print(f"\n===== {forest.upper()} =====", flush=True)
    model, dataset, temporal_loss, checkpoint, provenance, selected = load_site(forest)
    forest_gedi = {"Phase 1": [[], []], "Phase 2": [[], []]}
    forest_acc = {phase: {k: [] for k in ("target_mse", "abs_d1", "abs_d2", "gt2", "gt5")}
                  for phase in ("Phase 1", "Phase 2")}
    breakpoint_mse = {phase: {"breakpoint": [], "stable": []} for phase in ("Phase 1", "Phase 2")}

    for index in range(len(dataset)):
        sequence, target, meta = dataset[index]
        sequence = sequence.unsqueeze(0).to(DEVICE, non_blocking=True)
        target_b = target.unsqueeze(0).to(DEVICE, non_blocking=True)
        with torch.no_grad():
            out = model(sequence)
            z1, z2 = out[:, 0], out[:, 1]
            fitted, disturbance_mask, _ = temporal_loss.base.get_regression(out, target_b)
            aoi = out[:, 2] > 0.5

        arrays = {"Phase 1": z1, "Phase 2": z2}
        for phase, z in arrays.items():
            valid = aoi & torch.isfinite(z) & torch.isfinite(fitted)
            target_mse = ((z-fitted).square()[valid]).mean().item()

            d1 = z[:, 1:] - z[:, :-1]
            v1 = valid[:, 1:] & valid[:, :-1]
            d2 = z[:, 2:] - 2*z[:, 1:-1] + z[:, :-2]
            v2 = valid[:, 2:] & valid[:, 1:-1] & valid[:, :-2]
            abs_d1 = d1.abs()[v1]
            abs_d2 = d2.abs()[v2]
            vals = {
                "target_mse": target_mse,
                "abs_d1": abs_d1.mean().item(),
                "abs_d2": abs_d2.mean().item(),
                "gt2": (abs_d1 > 2).float().mean().item(),
                "gt5": (abs_d1 > 5).float().mean().item(),
            }
            for key, value in vals.items(): forest_acc[phase][key].append(value)
            crop_rows.append({"forest": forest, "crop_index": index, "phase": phase, **vals})

            bp = disturbance_mask.bool()
            per_pixel_mse = (z-fitted).square().mean(dim=1)
            spatial_valid = valid.all(dim=1)
            for label, mask in (("breakpoint", bp), ("stable", ~bp)):
                use = spatial_valid & mask
                if bool(use.any()): breakpoint_mse[phase][label].append(per_pixel_mse[use].mean().item())

            # Bounded sample for distribution plots; deterministic stride avoids huge files.
            sampled_d1 = abs_d1.detach().cpu().numpy()[::max(1, abs_d1.numel()//400)]
            sampled_d2 = abs_d2.detach().cpu().numpy()[::max(1, abs_d2.numel()//400)]
            change_rows.extend({"forest": forest, "phase": phase, "order": "|first difference|", "value": float(v)} for v in sampled_d1)
            change_rows.extend({"forest": forest, "phase": phase, "order": "|second difference|", "value": float(v)} for v in sampled_d2)

            gvalid = torch.isfinite(target_b) & (target_b != 0) & torch.isfinite(z)
            forest_gedi[phase][0].append(target_b[gvalid].detach().cpu().numpy())
            forest_gedi[phase][1].append(z[gvalid].detach().cpu().numpy())

        if (index + 1) % 20 == 0 or index + 1 == len(dataset):
            print(f"[{forest}] {index+1}/{len(dataset)} crops", flush=True)

    for phase in ("Phase 1", "Phase 2"):
        y = np.concatenate(forest_gedi[phase][0]); p = np.concatenate(forest_gedi[phase][1])
        gm = regression_metrics(y, p)
        gedi_rows.append({"forest": forest, "phase": phase, **gm})
        summary_rows.append({
            "forest": forest, "phase": phase, "n_crops": len(dataset),
            **{key: float(np.mean(values)) for key, values in forest_acc[phase].items()},
            "target_mse_breakpoint": float(np.mean(breakpoint_mse[phase]["breakpoint"])) if breakpoint_mse[phase]["breakpoint"] else np.nan,
            "target_mse_stable": float(np.mean(breakpoint_mse[phase]["stable"])) if breakpoint_mse[phase]["stable"] else np.nan,
            **gm,
            "product": selected["product"],
            "lambda_temp": float(selected["lambda_temp"]),
            "D_m": float(selected["drop_m"]),
            "K": int(selected["K"]),
            "checkpoint": str(checkpoint),
            "checkpoint_sha256": selected["checkpoint_sha256"],
        })
    del model, dataset, temporal_loss
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

summary = pd.DataFrame(summary_rows)
crops = pd.DataFrame(crop_rows)
changes = pd.DataFrame(change_rows)
gedi = pd.DataFrame(gedi_rows)
summary.to_csv(OUT / "phase1_phase2_temporal_summary_val.csv", index=False)
crops.to_csv(OUT / "phase1_phase2_temporal_crop_metrics_val.csv", index=False)
changes.to_csv(OUT / "phase1_phase2_temporal_change_samples_val.csv.gz", index=False, compression="gzip")
gedi.to_csv(OUT / "phase1_phase2_gedi_calibration_val.csv", index=False)
display(summary)

## Direct Phase 2 minus Phase 1 changes

Negative values are improvements for MSE, annual differences, MAE, RMSE, and absolute bias. For (R^2), slope and spread ratio, the direction must be interpreted relative to the ideal value of 1 rather than from the signed difference alone.

In [ ]:
wide = summary.pivot(index="forest", columns="phase")
delta = pd.DataFrame(index=wide.index)
for metric in ["target_mse", "abs_d1", "abs_d2", "gt2", "gt5", "mae", "rmse", "r2", "bias", "slope", "std_ratio"]:
    delta[f"delta_{metric}_P2_minus_P1"] = wide[(metric, "Phase 2")] - wide[(metric, "Phase 1")]
delta.to_csv(OUT / "phase2_minus_phase1_temporal_deltas_val.csv")
display(delta)

## Publication-oriented figures

In [ ]:
plt.style.use("default")
labels = {"ifran":"Ifran", "maamoura":"Maamoura", "agadir":"Agadir"}
palette = {"Phase 1":"#6b7280", "Phase 2":"#2563eb"}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.7), constrained_layout=True)
metrics_plot = [
    ("target_mse", "MSE to temporal target (m²)"),
    ("abs_d1", "Mean |annual change| (m)"),
    ("abs_d2", "Mean |second difference| (m)"),
    ("gt2", "Annual changes >2 m (%)"),
    ("gt5", "Annual changes >5 m (%)"),
]
plot_df = summary.copy()
plot_df["gt2"] *= 100
plot_df["gt5"] *= 100
x = np.arange(len(FORESTS), dtype=float)
width = 0.36
for ax, (metric, ylabel) in zip(axes, metrics_plot):
    for offset, phase in ((-width/2, "Phase 1"), (width/2, "Phase 2")):
        frame = plot_df[plot_df.phase.eq(phase)].set_index("forest").reindex(FORESTS)
        ax.bar(x + offset, frame[metric].to_numpy(float), width=width,
               color=palette[phase], label=phase)
    ax.set_xticks(x, [labels[f] for f in FORESTS], rotation=25, ha="right")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
axes[0].legend(title="", frameon=False)
fig.savefig(OUT / "phase1_phase2_temporal_metrics_val.png", dpi=350, bbox_inches="tight")
fig.savefig(OUT / "phase1_phase2_temporal_metrics_val.pdf", bbox_inches="tight")
plt.show()

def ecdf(values):
    values = np.sort(np.asarray(values, float))
    values = values[np.isfinite(values)]
    return values, np.arange(1, len(values)+1, dtype=float) / max(1, len(values))

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True, sharex="row")
for col, forest in enumerate(FORESTS):
    for row, order in enumerate(("|first difference|", "|second difference|")):
        ax = axes[row, col]
        for phase in ("Phase 1", "Phase 2"):
            values = changes.loc[
                changes.forest.eq(forest) & changes.order.eq(order) & changes.phase.eq(phase),
                "value",
            ].to_numpy(float)
            xx, yy = ecdf(values)
            ax.plot(xx, yy, color=palette[phase], lw=1.6, label=phase)
        ax.set_title(labels[forest] if row == 0 else "")
        ax.set_xlabel("Absolute annual difference (m)" if row == 0 else "Absolute second difference (m)")
        ax.set_ylabel("Cumulative proportion" if col == 0 else "")
axes[0, 0].legend(frameon=False)
fig.savefig(OUT / "phase1_phase2_temporal_difference_cdf_val.png", dpi=350, bbox_inches="tight")
fig.savefig(OUT / "phase1_phase2_temporal_difference_cdf_val.pdf", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), constrained_layout=True)
for ax, forest in zip(axes, FORESTS):
    frame = gedi[gedi.forest.eq(forest)].set_index("phase")
    for phase, marker in (("Phase 1", "o"), ("Phase 2", "s")):
        ax.scatter(frame.loc[phase,"slope"], frame.loc[phase,"std_ratio"], s=85, marker=marker,
                   color=palette[phase], label=phase, zorder=3)
    ax.plot(frame.loc[["Phase 1", "Phase 2"], "slope"],
            frame.loc[["Phase 1", "Phase 2"], "std_ratio"],
            color="#94a3b8", lw=1.5, zorder=1)
    ax.axvline(1, ls="--", color="black", lw=.8)
    ax.axhline(1, ls="--", color="black", lw=.8)
    ax.set_title(labels[forest])
    ax.set_xlabel("Regression slope")
    ax.set_ylabel("Predicted/observed SD ratio")
axes[0].legend(frameon=False)
fig.savefig(OUT / "phase1_phase2_calibration_val.png", dpi=350, bbox_inches="tight")
fig.savefig(OUT / "phase1_phase2_calibration_val.pdf", bbox_inches="tight")
plt.show()

## Interpretation guard

Phase 2 supports the temporal-coherence claim only if it reduces temporal-target MSE and implausibly large annual changes without materially degrading GEDI accuracy or collapsing slope and spread ratio. The notebook intentionally reports all criteria together; no single smoothness metric is treated as sufficient evidence.